# HEART FAILURE RISK - QLoRA fine-tuning on mistral 7B

In [1]:
!pip install -q unsloth datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 21.2 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 23.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 48.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 46.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 45.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 13.0 MB/s eta 0:

In [2]:
from unsloth import FastLanguageModel
import os
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, TextStreamer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
JSONL_PATH = "/kaggle/input/datasets/lightvvcx/heart-lora-jsonl/heart_lora_ready.jsonl"
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
OUTPUT_DIR = "/kaggle/working/heart-lora-final"

In [4]:
LORA_R = 16
LORA_ALPHA = 32
EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 2e-4
MAX_SEQ_LEN = 512

## Load dataset

In [5]:
dataset = load_dataset("json", data_files=JSONL_PATH, split="train")
print(f"dataset loaded at: {JSONL_PATH} with length: {len(dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

dataset loaded at: /kaggle/input/datasets/lightvvcx/heart-lora-jsonl/heart_lora_ready.jsonl with length: 399


In [6]:
def format_sample(sample):
    return {
        "text": (
            f"<|user|>\n{sample['instruction']}\n\n"
            f"{sample['input']}<|end|>\n"
            f"<|assistant|>\n{sample['output']}<|end|>"
        )
    }

In [7]:
dataset = dataset.map(format_sample)
print(f"dataset formatted: {dataset}")

Map:   0%|          | 0/399 [00:00<?, ? examples/s]

dataset formatted: Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 399
})


In [8]:
print(f"preview: {dataset[10]['text']}")

preview: <|user|>
You are a clinical decision support assistant specialized in heart failure risk assessment. Given a patient's clinical profile in natural language, assess their risk of mortality, explain the key contributing factors, and clearly flag if any critical clinical information is missing that would affect the confidence of your assessment.

Patient is a 70-year-old male. He has a known medical history of diabetes. He is a non-smoker. Ejection fraction is 40.0%, which is mildly below normal. This is slightly under the normal range of 55–70% and warrants monitoring. Creatinine phosphokinase (CPK) level is 2695 mcg/L, which is significantly elevated. Levels this high may indicate serious heart muscle damage or stress. Serum creatinine is 1.0 mg/dL, which is normal. Within normal range, suggesting adequate kidney function. Serum sodium is 137.0 mEq/L, which is normal. Within the healthy electrolyte range. Platelet count is 241,000, which is normal. Within the expected range. Th

## Load Model

In [9]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None
)

==((====))==  Unsloth 2026.3.10: Fast Mistral patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

## Attach LoRA adapters

In [10]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    target_modules=['qkv_proj', "o_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42
)
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: You added custom modules, but Unsloth hasn't optimized for this.
Beware - your finetuning might be noticeably slower!


Unsloth 2026.3.10 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


## Train

In [11]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=10,
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

In [12]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/399 [00:00<?, ? examples/s]

In [13]:
trainer.model_accepts_loss_kwargs = False

In [14]:
print("start training")
trainer.train()
print("training complete!")

start training


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 399 | Num Epochs = 3 | Total steps = 39
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 3,145,728 of 3,824,225,280 (0.08% trained)


Step,Training Loss
10,0.425586
20,0.374595
30,0.245319


training complete!


## Save adapter

In [25]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

('/kaggle/working/heart-lora-final/tokenizer_config.json',
 '/kaggle/working/heart-lora-final/chat_template.jinja',
 '/kaggle/working/heart-lora-final/tokenizer.json')

In [26]:
print(f"Adapter saved to {OUTPUT_DIR}")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"   {f}  ({size:.1f} MB)")

Adapter saved to /kaggle/working/heart-lora-final
   README.md  (0.0 MB)
   tokenizer_config.json  (0.0 MB)
   adapter_model.safetensors  (12.6 MB)
   adapter_config.json  (0.0 MB)
   chat_template.jinja  (0.0 MB)
   tokenizer.json  (3.6 MB)


## Inference

In [27]:
from peft import PeftModel
from unsloth import FastLanguageModel

In [31]:
model_inf, tokenizer_inf = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3-mini-4k-instruct",
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

==((====))==  Unsloth 2026.3.10: Fast Mistral patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [32]:
model_inf = PeftModel.from_pretrained(model_inf, OUTPUT_DIR)

FastLanguageModel.for_inference(model_inf)
model_inf.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
              (k_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
              (v_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (defau

In [33]:
INSTRUCTION = (
    "You are a clinical decision support assistant specialized in heart failure risk assessment. "
    "Given a patient's clinical profile in natural language, assess their risk of mortality, "
    "explain the key contributing factors, and clearly flag if any critical clinical information "
    "is missing that would affect the confidence of your assessment."
)

In [34]:
def assess_patient(description):
    prompt = (
        f"<|user|>\n{INSTRUCTION}\n\n"
        f"{description}<|end|>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer_inf(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN
    ).to("cuda")

    with torch.no_grad():
        output = model_inf.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=256,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer_inf.eos_token_id,
            use_cache=False,              
        )                                 

    return tokenizer_inf.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

In [37]:
test_patients = [
    (
        "Complete - high risk",
        "Patient is a 72-year-old male with high blood pressure and diabetes. "
        "Non-smoker. Ejection fraction is 22%, serum creatinine is 2.4 mg/dL, "
        "serum sodium is 128 mEq/L, CPK is 1400 mcg/L, platelet count 180,000. "
        "Follow-up: 30 days."
    ),
    (
        "Incomplete - should flag missing info",
        "Patient is a 55-year-old female with diabetes. "
        "Ejection fraction is 35%. Non-smoker."
    ),
    (
        "Complete - low risk",
        "Patient is a 45-year-old female. No history of anaemia, diabetes, or high blood pressure. "
        "Non-smoker. Ejection fraction 62%, serum creatinine 0.9 mg/dL, "
        "serum sodium 140 mEq/L, CPK 120 mcg/L, platelets 250,000. Follow-up: 200 days."
    ),
]

In [38]:
for label, description in test_patients:
    print("=" * 60)
    print(f"TEST: {label}")
    print("=" * 60)
    print("RESPONSE:", assess_patient(description))
    print()

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST: Complete - high risk


Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESPONSE: This patient is at high risk of mortality. Key contributing factors include severely reduced ejection fraction, elevated serum creatinine, mildly low serum sodium, and elevated CPK. Outcome: patient was lost to follow-up.

TEST: Incomplete - should flag missing info


Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESPONSE: This patient is at high risk of mortality. The following factors contributed to this assessment:

- Serious clinical risk factors:
  - Ejection fraction (35%): significantly below the normal range (55–70%), indicating severe cardiac dysfunction.
  - Diabetes: a known risk factor for heart failure.

- Non-critical risk factors:
  - Non-smoker: a protective factor compared to smoking.

Please note that the patient did not provide their creatinine phosphokinase (CPK) level, which is a critical marker for cardiac health. Without this information, the confidence of the assessment is reduced.

TEST: Complete - low risk
RESPONSE: This patient appears to be at a low risk of mortality from heart failure. Key contributing factors include a healthy ejection fraction (within normal range), normal serum creatinine (indicating normal kidney function), and normal serum sodium (electrolyte balance). The patient's clinical profile is well within normal ranges, with no critical risk factors de